# Defining Boundary Conditions

With the mesh completed, the next step is to define the values of the fields at the boundary of the mesh.

# Libraries / Packages

In [3]:
import numpy as np
import meshio
from dolfinx import *

# Variables & Parameters

In [5]:
# Put variables here

# Input Pre-Defined Mesh

This is collimator.msh file that was defined in gmsh_collimator.ipynb. GMSH is separate to FEniCSx, and FEniCSx requires the mesh to be in a xmdf format. The file is converted using meshio. 

In [ ]:
"""
Convert a Gmsh .msh file to XDMF for use in DOLFINx, using only `meshio`
(no `gmsh` Python module / binary required).

Produces two files:
  - <name>_mesh.xdmf   : the 3D volume mesh (tetrahedra) with cell tags
                         for the volume physical groups (e.g. collimator,
                         anode, cathode, trigger)
  - <name>_facets.xdmf : the 2D facet mesh (triangles) with cell tags for
                         the surface physical groups (e.g.
                         collimator_surface, anode_surface, ...)

Load both in dolfinx with dolfinx.io.XDMFFile, e.g.:

    from mpi4py import MPI
    from dolfinx.io import XDMFFile

    with XDMFFile(MPI.COMM_WORLD, "collimator_mesh.xdmf", "r") as xdmf:
        mesh = xdmf.read_mesh(name="Grid")
        cell_tags = xdmf.read_meshtags(mesh, name="Grid")

    mesh.topology.create_connectivity(mesh.topology.dim - 1, mesh.topology.dim)
    with XDMFFile(MPI.COMM_WORLD, "collimator_facets.xdmf", "r") as xdmf:
        facet_tags = xdmf.read_meshtags(mesh, name="Grid")
"""

import sys
import meshio


def create_sub_mesh(mesh: meshio.Mesh, cell_type: str) -> meshio.Mesh:
    """Extract all cells of `cell_type` plus their 'gmsh:physical' tags,
    dropping unused points so the sub-mesh is self-contained."""
    cells = mesh.get_cells_type(cell_type)
    physical_tags = mesh.get_cell_data("gmsh:physical", cell_type)

    # meshio keeps the full point array; XDMF wants only points that are
    # actually referenced, re-indexed from 0.
    used_ids = sorted(set(cells.flatten()))
    id_map = {old: new for new, old in enumerate(used_ids)}
    new_cells = [[id_map[i] for i in tri] for tri in cells]
    new_points = mesh.points[used_ids]

    return meshio.Mesh(
        points=new_points,
        cells={cell_type: new_cells},
        cell_data={"name_to_read": [physical_tags]},
    )


def main(msh_path: str, out_prefix: str) -> None:
    print(f"Reading {msh_path} ...")
    mesh = meshio.read(msh_path)

    # Gmsh always writes the highest-dimension elements last/most-refined;
    # for a 3D volume mesh this is 'tetra' cells and boundary/interface
    # tags live on 'triangle' cells.
    volume_mesh = create_sub_mesh(mesh, "tetra")
    facet_mesh = create_sub_mesh(mesh, "triangle")

    vol_path = f"{out_prefix}_mesh.xdmf"
    facet_path = f"{out_prefix}_facets.xdmf"

    meshio.write(vol_path, volume_mesh)
    meshio.write(facet_path, facet_mesh)

    print(f"Wrote volume mesh  -> {vol_path} ({vol_path[:-5]}.h5)")
    print(f"  physical tags present: {sorted(set(volume_mesh.cell_data['name_to_read'][0]))}")
    print(f"Wrote facet mesh   -> {facet_path} ({facet_path[:-5]}.h5)")
    print(f"  physical tags present: {sorted(set(facet_mesh.cell_data['name_to_read'][0]))}")


if __name__ == "__main__":
    msh_file = sys.argv[1] if len(sys.argv) > 1 else "collimator.msh"
    prefix = sys.argv[2] if len(sys.argv) > 2 else "collimator"
    main(msh_file, prefix)


In [ ]:
msh = meshio.read("collimator.msh")
tetra_cells = []
triangle_cells = []
for cell in msh.cells:
    if  cell.type == "tetra":
        if len(tetra_cells) == 0:
            tetra_cells = cell.data
        else:
            tetra_cells = np.vstack([tetra_cells, cell.data])
    elif cell.type == "triangle":
        if len(triangle_cells) == 0:
            triangle_cells = cell.data
        else:
            triangle_cells = np.vstack([triangle_cells, cell.data])

tetra_mesh = meshio.Mesh(points=msh.points, cells={"tetra": tetra_cells})
meshio.write("collimator_tetra.xdmf", tetra_mesh)
triangle_mesh = meshio.Mesh(points=msh.points, cells={"triangle": triangle_cells})
meshio.write("collimator_triangle.xdmf", triangle_mesh)

In [ ]:
# CODE WRITTEN BY ME THAT DIDN'T WORK


#mesh = meshio.Mesh("collimator_tetra.xmdf", cells=triangle)

# import meshio
# msh = meshio.read("collimator.msh")
# for cell in msh.cells:
#     triangle_cells = cell.data
# for key in msh.cell_data_dict["gmsh:physical"].keys():
#     if key == "triangle":
#         triangle_data = msh.cell_data_dict["gmsh:physical"][key]
# mesh = meshio.Mesh(points=msh.points,
#                            cells=[("triangle", triangle_cells)],
#                            cell_data={"name_to_read":[triangle_data]})
# meshio.write("collimator.xdmf",mesh)

# import dolfinx
# from dolfinx.io import XDMFFile

# with dolfinx.io.XDMFFile(MPI.COMM_WORLD, "collimator.xdmf", "r") as xdmf:
#        mesh = xdmf.read_mesh(name="Grid")

In [ ]:
# AI-GENERATED SUGGESTIONS - reading into FEniCSx. (Doesn't work - bad_alloc error on section 2, section 1 relies on gmsh which is not available in this environment.)

"""
Reading collimator.msh into FEniCSx (dolfinx).

Two approaches are shown:
  1. dolfinx.io.gmshio  -- reads the .msh file directly, recommended.
     Physical groups you defined in gmsh (e.g. "collimator", "anode",
     "magnets_surface", ...) come through as integer tags on cells and facets.
  2. meshio -> XDMF -> dolfinx -- older / more portable route, useful if you
     need to inspect or post-process the mesh with meshio first, or if you're
     on a dolfinx version without gmshio.
"""

from mpi4py import MPI
import dolfinx
from dolfinx.io import gmshio

# ---------------------------------------------------------------------------
# Approach 1: read the .msh file directly with dolfinx.io.gmshio
# ---------------------------------------------------------------------------

# # gdim=3 because this is a 3D volume mesh (collimator/anode/cathode/trigger/magnets)
# mesh, cell_tags, facet_tags = gmshio.read_from_msh(
#     "collimator.msh",
#     MPI.COMM_WORLD,
#     rank=0,
#     gdim=3,
# )

# print(f"Mesh has {mesh.topology.index_map(mesh.topology.dim).size_local} cells "
#       f"on rank {MPI.COMM_WORLD.rank}")

# # cell_tags holds the physical VOLUME groups you defined, e.g.:
# #   phys_col   = 1  "collimator"
#   phys_a     = 2  "anode"
#   phys_c     = 4  "cathode"
#   phys_trig  = 5  "trigger"
# Use these to restrict integrals / material properties to a subdomain, e.g.:
#   collimator_cells = cell_tags.find(1)

# facet_tags holds the physical SURFACE groups, e.g.:
#   phys_surf_col = 7   "collimator_surface"
#   phys_surf_a   = 8   "anode_surface"
#   phys_surf_c   = 9   "cathode_surface"
#   phys_surf_trig = 10 "trigger_surface"
#   phys_surf_mag  = 11 "magnets_surface"
# Use these for boundary conditions, e.g.:
#   magnet_facets = facet_tags.find(11)

# # Save mesh + tags to XDMF so you can reload without re-parsing gmsh output,
# # and so you can visualise cell_tags/facet_tags in ParaView.
# with dolfinx.io.XDMFFile(MPI.COMM_WORLD, "collimator.xdmf", "w") as xdmf:
#     xdmf.write_mesh(mesh)
#     mesh.topology.create_connectivity(mesh.topology.dim - 1, mesh.topology.dim)
#     xdmf.write_meshtags(cell_tags, mesh.geometry)
#     xdmf.write_meshtags(facet_tags, mesh.geometry)

# # ---------------------------------------------------------------------------
# # Approach 2: meshio -> XDMF -> dolfinx (fallback / manual route)
# ---------------------------------------------------------------------------
# Uncomment to use instead of gmshio. This is the route your original script
# was heading towards with `msh = meshio.read(meshname)`.

# import os
# for f in ["collimator_volume.xdmf", "collimator_volume.h5",
#           "collimator_boundary.xdmf", "collimator_boundary.h5"]:
#     if os.path.exists(f):
#         os.remove(f)

# print("removed old files")

# import meshio

# def create_meshio_xdmf(msh_file, cell_type, prune_z=False):
#     msh = meshio.read(msh_file, file_format="gmsh")
#     cells = msh.get_cells_type(cell_type)
#     cell_data = msh.get_cell_data("gmsh:physical", cell_type)
#     points = msh.points[:, :2] if prune_z else msh.points
#     return meshio.Mesh(
#         points=points,
#         cells={cell_type: cells},
#         cell_data={"tags": [cell_data]},
#     )

# msh = meshio.read("collimator.msh")

# # Volume mesh (tetrahedra) -> for cell_tags
# tet_mesh = create_meshio_xdmf("collimator.msh", "tetra")
# meshio.write("collimator_volume.xdmf", tet_mesh)
# print("created collimator_volume.xdmf")

# # Boundary mesh (triangles) -> for facet_tags
# tri_mesh = create_meshio_xdmf("collimator.msh", "triangle")
# meshio.write("collimator_boundary.xdmf", tri_mesh)
# print("created collimator_boundary.xdmf")

# with dolfinx.io.XDMFFile(MPI.COMM_WORLD, "collimator_volume.xdmf", "r") as xdmf:
#     mesh = xdmf.read_mesh(name="Grid")
#     print("read mesh")
#     cell_tags = xdmf.read_meshtags(mesh, name="Grid")
#     print("cell_tags done")

# mesh.topology.create_connectivity(mesh.topology.dim - 1, mesh.topology.dim)
# with dolfinx.io.XDMFFile(MPI.COMM_WORLD, "collimator_boundary.xdmf", "r") as xdmf:
#     facet_tags = xdmf.read_meshtags(mesh, name="Grid")

removed old files

created collimator_volume.xdmf
created collimator_boundary.xdmf
read mesh


MemoryError: std::bad_alloc